# Benchmarking

In [29]:
import numpy as np
import jax
import jax.numpy as jnp
import time
import WDM

We will time some of the dwt methods and their inverses on a time series of length $N=2^{14}=16384$. We will also show that they agree exactly. 

In [30]:
wdm = WDM.WDM.WDM_transform(dt=1., 
                            Nf=2**8, 
                            N=2**14)

x = np.random.normal(size=wdm.N) # white noise

In [31]:
t0 = time.time()
w = wdm.forward_transform_exact(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 4627.18 milliseconds


In [32]:
t0 = time.time()
x_recovered = wdm.inverse_transform_exact(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

max_error = jnp.max(jnp.abs(x - x_recovered))

relative_error = (
    jnp.linalg.norm(x - x_recovered)
    / jnp.linalg.norm(x)
)
print(f"Maximum error:  {max_error:.3e}")
print(f"Relative error: {relative_error:.3e}")

Time taken: 4097.76 milliseconds
Maximum error:  8.013e-12
Relative error: 1.513e-12


Key Note: You might notice the maximum and relative error for the exact method is higher than the fft method. This is due to the fact that this approach uses `window_TD` (`wdm.inverse_transform_truncated_window` also uses this). `window_FD` is more localised and hence more accurate, with a lower floor. 

In [33]:
t0 = time.time()
w = wdm.forward_transform_short_fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 226.11 milliseconds


The short FFT method, has no inverse call. 

The `forward_transform_fft` method is the one used in production when `wdm.dwt` is called. 

In [34]:
t0 = time.time()
w = wdm.forward_transform_fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 86.40 milliseconds


In [35]:
t0 = time.time()
x_recovered = wdm.inverse_transform_fft(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

max_error = jnp.max(jnp.abs(x - x_recovered))

relative_error = (
    jnp.linalg.norm(x - x_recovered)
    / jnp.linalg.norm(x)
)
print(f"Maximum error:  {max_error:.3e}")
print(f"Relative error: {relative_error:.3e}")

Time taken: 104.48 milliseconds
Maximum error:  2.831e-15
Relative error: 6.564e-16


Compilation means subsequent calls can be much faster.

In [36]:
t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 4.26 milliseconds
Time taken: 1.20 milliseconds


Same thing holds for an inverse transform:

In [37]:
t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 1.72 milliseconds
Time taken: 0.99 milliseconds


Vectorisation means that batched transforms can also be much faster.

In [38]:
x = np.random.normal(size=(20, wdm.N)) # white noise

t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 74.94 milliseconds
Time taken: 4.82 milliseconds


As above, same thing holds for the inverse transform:

In [39]:
t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 124.01 milliseconds
Time taken: 6.95 milliseconds


Let's compare this with the cost of an FFT on the same set of time series.

In [40]:
t0 = time.time()
w = jnp.fft.fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
w = jnp.fft.fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 7.58 milliseconds
Time taken: 5.26 milliseconds


The new inverse and forward transform is also independent of $q$ as a paremeter.

In [41]:
x = np.random.normal(size=wdm.N) # white noise

q_vals = [2, 4, 8, 16]

for q in q_vals:
    wdm_q = WDM.WDM.WDM_transform(dt=1., 
                            Nf=2**8, 
                            N=2**14,
                            q = q,)

    w_q = wdm_q.forward_transform_fft(x)

    # Compile first
    wdm_q.inverse_transform_fft(w_q).block_until_ready()

    print(f"q = {q}")

    print("fast FFT inverse:")
    %timeit wdm_q.inverse_transform_fft(w_q).block_until_ready()

    print()

q = 2
fast FFT inverse:
403 μs ± 13.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

q = 4
fast FFT inverse:
374 μs ± 30.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

q = 8
fast FFT inverse:
386 μs ± 19.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

q = 16
fast FFT inverse:
368 μs ± 18.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

